In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# DN = 'C://work/dev/python/progs/texts/sec_bert/'
DN = '/home/jovyan/work/sec_bert/'

import os
os.chdir(DN)

In [ ]:
import nltk
nltk.download('punkt'),  nltk.download('punkt_tab')

# Train

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from itertools import chain
import click
import os

from ruamel.yaml import YAML

# Загрузка данных

In [ ]:
from src.train import load_external_data, enc_classes

# получаем данные митра
conf = YAML().load(open('params.yaml'))
conf_bert = YAML().load(open('dvc_pipes/bert/params_bert.yaml'))
conf_bert_ttp = YAML().load(open('dvc_pipes/bert_ttp/params_bert_ttp.yaml'))


In [ ]:
df = load_external_data(conf)
df, mlb, mlb_ttp = enc_classes(df, conf, use_rare_ttp=False)

# на самом деле 208 train тут уже есть - синтетика
df['split'] = df['split'].fillna('tr')

# train

In [ ]:
from src.train import train

train()

In [ ]:
import torch

model_bert_ttp = torch.load(conf['train_fin']['model_technik_fn'])
model_bert = torch.load(conf['train_fin']['model_taktic_fn'])
    
thresh_ttp_l = joblib.load(conf['train_fin']['thresh_ttp_fn'])
thresh_l = joblib.load(conf['train_fin']['thresh_fn'])

## nttp pred

In [ ]:
from src.predict import predict

pred_df = predict(df[['sentence']].assign(target=1), model_bert, thresh_l, conf_bert, suf='labels')
pred_df.head()

In [ ]:

pred_true_df = pred_df.copy().assign(target=df.target)

In [ ]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_labels'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_labels'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

## ttp pred

In [ ]:
conf_bert_ttp = YAML().load(open('dvc_pipes/bert_ttp/params_bert_ttp.yaml'))
conf_bert_ttp['nn'] = conf_bert_ttp['nn_ttp']
conf_bert_ttp['nn_bert'] = conf_bert_ttp['nn_bert_ttp']

In [ ]:
pred_df = predict(pred_df, model_bert_ttp, thresh_ttp_l, conf_bert_ttp, suf='ttp')
pred_df.head()

In [ ]:
pred_true_df = pred_df.copy().assign(target=df.target_ttp)


In [ ]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_ttp'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(pred_true_df['target'].values.tolist()), 
                                                    np.array(pred_true_df['pred_ttp'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

# Добавим полей с метками


In [ ]:
pred_df['pred_str_ttp'] = pred_df['pred_ttp'].map(lambda x: mlb_ttp.inverse_transform(np.array([x]))[0])
pred_df['pred_str_labels'] = pred_df['pred_labels'].map(lambda x: mlb.inverse_transform(np.array([x]))[0])

pred_df = pred_df.assign(true_labels=df.labels, true_ttp=df.ttp)

In [ ]:
pred_df